# Small Dataset Tuning

小規模 HDF5 を使って、GNN の設定を Jupyter 上で変更しながら試すための notebook です。学習本体は script 実行と同じ `configs/small_tuning_example.json` と `talesd_gnn_reconstruction.tuning` を使うので、notebook で試した設定をそのまま script 側にも移せます。

基本的に触る場所は次の2か所です。

- `DATASET_KEY`: 使う小規模データセットを `flat2000`, `flat500`, `flat200` から選びます。
- `overrides`: epoch 数、batch size、learning rate、モデル幅などの実験パラメーターを変えます。

## 1. 初期化

このセルはリポジトリの場所を見つけて、共通の tuning 関数を import します。通常ここは変更しません。Jupyter を DiCOS App から開いた場合でも、作業リポジトリが current directory でない場合は `/ceph/sharedfs/.../talesd_gnn_reconstruction` を使います。

In [ ]:
from pathlib import Path
import json
import sys

repo = Path.cwd()
if not (repo / "src" / "talesd_gnn_reconstruction").exists():
    repo = Path("/ceph/sharedfs/work/SATORI/ikomae/src/talesd_gnn_reconstruction")
sys.path.insert(0, str(repo / "src"))

from talesd_gnn_reconstruction.tuning import apply_overrides, load_config, run_training_from_config

config_path = repo / "configs" / "small_tuning_example.json"
base_config = load_config(config_path)
print(f"repo: {repo}")
print(f"config: {config_path}")

## 2. データセットを選ぶ

ここがデータサイズを切り替える場所です。`DATASET_KEY` だけを変更します。

- `flat2000`: 最大の小規模データセット。最終確認に近い条件で使います。
- `flat500`: パラメーター探索用。速度と統計量のバランスを見るために使います。
- `flat200`: 軽い確認用。コードや設定が動くかを早く見るために使います。

`SMALL_GRAPH_SETS` は HDF5 shard が入っているディレクトリです。各ディレクトリ内の `*.h5` は学習時に自動で展開されます。

In [ ]:
SMALL_GRAPH_SETS = {
    "flat2000": "/dicos_ui_home/ikomae/work/gnn/graphs/flat2000",
    "flat500": "/dicos_ui_home/ikomae/work/gnn/graphs/flat500",
    "flat200": "/dicos_ui_home/ikomae/work/gnn/graphs/flat200",
}

# ここだけ変える: "flat2000", "flat500", "flat200"
DATASET_KEY = "flat2000"

if DATASET_KEY not in SMALL_GRAPH_SETS:
    raise ValueError(f"unknown DATASET_KEY: {DATASET_KEY}. choose from {sorted(SMALL_GRAPH_SETS)}")

graph_dir = Path(SMALL_GRAPH_SETS[DATASET_KEY]).expanduser()
dataset_overrides = {
    "graphs": str(graph_dir),
    "small_dataset.output": str(graph_dir),
    "run_name": f"small_{DATASET_KEY}_reco_quality_tuning",
    "config_name": f"small_{DATASET_KEY}_reco_quality_physics_wfcnn-gru_h128_l4",
}
config = apply_overrides(base_config, dataset_overrides)

if graph_dir.exists():
    h5_files = sorted(graph_dir.glob("*.h5"))
    print(f"dataset: {DATASET_KEY}")
    print(f"graph_dir: {graph_dir}")
    print(f"h5 shards: {len(h5_files)}")
    print("preview:")
    for path in h5_files[:5]:
        print(f"  {path.name}")
else:
    print(f"dataset: {DATASET_KEY}")
    print(f"graph_dir: {graph_dir}")
    print("この kernel からは graph_dir が見えていません。DiCOS 側で実行しているか確認してください。")

## 3. 学習パラメーターを変える

`overrides` に書いた値だけが JSON 設定を上書きします。キーは `train.batch_size` のように `.` で階層を指定します。

よく触る候補は次です。

- `train.epochs`: 最大 epoch 数。小規模 tuning では短めから確認します。
- `train.batch_size`: GPU メモリと速度に効きます。大きすぎると落ちます。
- `train.learning_rate`: 最初に見るべき主要パラメーターです。
- `train.hidden_dim`, `train.num_layers`: モデル容量です。大きくすると遅くなり、過学習もしやすくなります。
- `train.dropout`, `train.weight_decay`: 正則化です。
- `train.training_task`, `train.mass_classification`, `train.quality_prediction`: 何を学習するかを切り替えます。

このセルではまだ学習は走りません。`dry_run=True` で、使われる graph と checkpoint path だけを確認します。

In [ ]:
overrides = {
    "run_name": f"small_{DATASET_KEY}_reco_quality_notebook_trial",
    "config_name": f"small_{DATASET_KEY}_reco_quality_notebook_trial",
    "train.epochs": 4,
    "train.batch_size": 128,
    "train.learning_rate": 5e-4,
    "train.hidden_dim": 128,
    "train.num_layers": 4,
    "train.dropout": 0.05,
    "train.device": "auto",
    "train.save_diagnostics": False,
}

trial_config = apply_overrides(config, overrides)
resolved = run_training_from_config(trial_config, config_path=config_path, dry_run=True)
print(json.dumps({
    "dataset": DATASET_KEY,
    "graphs": resolved["graphs"][:5],
    "graph_count": len(resolved["graphs"]),
    "run_dir": resolved["run_dir"],
    "checkpoint": resolved["checkpoint"],
    "train_kwargs": resolved["train_kwargs"],
}, indent=2)[:8000])

## 4. 学習を実行する

下のセルのコメントを外すと実際に学習が始まります。まずは `flat200` または `flat500` で短い epoch の確認を行い、設定が問題なければ `flat2000` に切り替えます。

出力先は上の dry run に表示された `run_dir` です。実行時には `run_dir/config/small_tuning_resolved.json` に、実際に使われた graph path と学習引数が保存されます。

In [ ]:
# result = run_training_from_config(trial_config, config_path=config_path)
# result